# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 3: Evaluation, Baselines, Traditional ML

Today we'll write some simple models to predict the price of a product

We'll use an approach to evaluate the performance of the model

And we'll test some Baseline Models using Traditional machine learning

In [14]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import Tester
from pricer.items import Item

In [4]:
LITE_MODE = False

In [6]:
username = "ed-donner"

dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

Generating train split:   0%|          | 0/800000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [7]:
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [8]:
def random_pricer(item):
    return random.randrange(1, 1000)

In [9]:
random.seed(42)
Tester.evaluate(random_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$104 $539 $29 $690 $252 $21 $85 $72 $719 $225 $20 $380 $894 $505 $11 $572 $354 $17 $179 $23 $90 $115 $433 $442 $304 $122 $291 $714 $567 $639 $539 $370 $66 $380 $489 $534 $769 $835 $207 $740 $626 $84 $680 $178 $129 $260 $142 $189 $836 $580 $310 $25 $380 $270 $47 $234 $861 $313 $417 $259 $591 $33 $657 $361 $79 $38 $757 $500 $263 $5 $534 $284 $570 $625 $584 $871 $759 $361 $575 $178 $602 $60 $17 $579 $207 $732 $115 $224 $756 $193 $866 $9 $370 $250 $456 $423 $821 $217 $103 $195 $264 $98 $650 $135 $470 $842 $675 $264 $43 $325 $591 $138 $516 $619 $56 $2 $449 $369 $221 $845 $640 $616 $501 $59 $502 $273 $844 $688 $616 $81 $164 $705 $52 $795 $259 $396 $70 $2 $89 $798 $902 $331 $818 $716 $129 $186 $627 $2 $141 $873 $918 $553 $423 $7 $253 $136 $204 $707 $255 $502 $19 $739 $557 $426 $80 $575 $39 $321 $185 $132 $497 $484 $326 $751 $53 $733 $85 $64 $549 $71 $158 $672 $133 $362 $16 $373 $258 $544 $420 $515 $223 $944 $532 $743 $866 $68 $527 $459 $67 $673 

In [15]:
training_prices = [item.price for item in train]
training_average = sum(training_prices)/len(training_prices)
print(training_average)

140.56967545


In [18]:
def constant_pricer(item):
    return training_average

Tester.evaluate(constant_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$78 $25 $86 $71 $111 $89 $4 $75 $105 $189 $572 $238 $121 $86 $61 $108 $61 $91 $70 $22 $7 $17 $56 $34 $191 $312 $354 $121 $42 $61 $121 $81 $19 $60 $25 $678 $81 $85 $73 $103 $59 $61 $106 $114 $79 $116 $123 $109 $5 $61 $105 $11 $334 $21 $87 $6 $134 $101 $62 $129 $95 $63 $50 $31 $488 $51 $99 $304 $16 $65 $109 $124 $139 $122 $91 $105 $16 $131 $124 $122 $21 $129 $111 $42 $114 $81 $42 $165 $21 $95 $119 $46 $121 $106 $132 $88 $107 $17 $129 $434 $41 $24 $104 $2 $108 $23 $116 $259 $110 $158 $81 $174 $110 $12 $55 $29 $116 $121 $85 $38 $125 $52 $70 $25 $59 $81 $121 $42 $38 $2 $69 $3 $55 $111 $76 $126 $64 $71 $12 $2 $76 $109 $60 $121 $53 $109 $96 $369 $124 $108 $122 $35 $94 $1 $121 $138 $92 $85 $179 $91 $148 $115 $99 $128 $699 $118 $307 $91 $101 $131 $116 $119 $279 $118 $39 $8 $113 $48 $46 $48 $514 $116 $159 $108 $91 $119 $7 $74 $81 $114 $106 $90 $106 $2 $41 $61 $29 $140 $90 $115 

In [20]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [21]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

In [22]:
train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [23]:
# Traditional Linear Regression

np.random.seed(42)

# Separate features and target
feature_columns = ["weight", "weight_unknown", "text_length"] 

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Train a Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")

print(f"Intercept: {model.intercept_}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean squared Error: {mse}")
print(f"R-squared Score: {r2}")

weight: 0.4486789805756245
weight_unknown: -6.627998877825423
text_length: 0.2469451895563049
Intercept: 51.11099822544172
Mean squared Error: 25615.843488572413
R-squared Score: -0.05946814914318366


In [24]:
def linear_regression_pricer(item):
    features = get_features(item)
    feature_df = pd.DataFrame([features])
    return model.predict(feature_df)[0]

In [25]:
Tester.evaluate(linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$55 $29 $87 $90 $92 $60 $1 $78 $104 $192 $556 $228 $141 $81 $50 $124 $69 $83 $50 $8 $10 $9 $61 $2 $175 $298 $345 $114 $34 $55 $100 $77 $10 $38 $24 $676 $78 $94 $50 $90 $48 $55 $83 $102 $82 $119 $100 $106 $2 $52 $106 $9 $353 $28 $92 $26 $136 $110 $37 $121 $97 $69 $43 $17 $460 $41 $83 $288 $4 $86 $109 $101 $139 $113 $99 $103 $8 $122 $122 $111 $25 $107 $98 $10 $108 $63 $60 $164 $33 $82 $109 $41 $101 $94 $114 $104 $97 $9 $151 $426 $37 $11 $135 $5 $100 $25 $107 $277 $102 $182 $90 $169 $135 $11 $45 $38 $100 $119 $98 $21 $125 $72 $76 $32 $35 $60 $109 $61 $10 $2 $78 $12 $58 $95 $58 $106 $65 $65 $1 $8 $61 $110 $54 $107 $33 $91 $93 $364 $107 $92 $118 $44 $76 $19 $111 $122 $102 $61 $138 $76 $135 $116 $83 $113 $710 $120 $269 $101 $107 $125 $110 $124 $275 $101 $52 $4 $119 $26 $48 $53 $481 $141 $139 $64 $102 $116 $19 $77 $78 $103 $94 $99 $122 $1 $30 $65 $25 $133 $112 $113 

In [26]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [27]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X= vectorizer.fit_transform(documents)

In [28]:
# Here are the 1,000 most common words that it picked, not including "stop words"

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len (selected_words)}")
print("Selected words:", selected_words)

Number of selected words: 2000
Selected words: ['000' '0l' '10' ... 'zinc' 'zipper' 'zoom']


In [29]:
regressor = LinearRegression()
regressor.fit(X, prices)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [30]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [31]:
Tester.evaluate(natural_language_linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$67 $124 $55 $6 $179 $211 $52 $50 $73 $7 $535 $186 $101 $150 $67 $96 $58 $49 $37 $29 $15 $87 $28 $9 $274 $241 $141 $13 $56 $80 $78 $57 $10 $43 $113 $418 $15 $65 $140 $71 $156 $61 $69 $26 $94 $59 $39 $49 $4 $13 $41 $91 $156 $31 $80 $69 $81 $166 $24 $12 $44 $18 $55 $24 $422 $93 $3 $312 $62 $212 $19 $33 $11 $130 $1 $36 $115 $36 $14 $130 $85 $59 $43 $42 $36 $117 $42 $105 $31 $177 $4 $106 $80 $27 $38 $91 $61 $1 $162 $102 $49 $56 $37 $15 $29 $4 $43 $214 $31 $98 $20 $20 $171 $42 $6 $149 $112 $25 $24 $40 $11 $108 $87 $28 $80 $27 $32 $127 $86 $29 $74 $40 $141 $23 $70 $55 $39 $103 $126 $76 $34 $171 $78 $95 $68 $91 $25 $235 $63 $32 $19 $189 $24 $27 $62 $49 $124 $28 $7 $9 $76 $26 $42 $13 $470 $22 $76 $50 $34 $9 $3 $23 $279 $27 $41 $144 $10 $40 $45 $144 $372 $18 $20 $13 $158 $32 $44 $16 $103 $36 $29 $122 $9 $27 $43 $60 $53 $6 $48 $18 

In [32]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

## Random Forest model

The Random Forest is a type of "**ensemble**" algorithm, meaning that it combines many smaller algorithms to make better predictions.

It uses a very simple kind of machine learning algorithm called a **decision tree**. A decision tree makes predictions by examining the values of features in the input. Like a flow chart with IF statements. Decision trees are very quick and simple, but they tend to overfit.

In our case, the "features" are the elements of the Vector - in other words, it's the number of times that a particular word appears in the product description.

So you can think of it something like this:

**Decision Tree**  
\- IF the word "TV" appears more than 3 times THEN  
-- IF the word "LED" appears more than 2 times THEN  
--- IF the word "HD" appears at least once THEN  
---- Price = $500


With Random Forest, multiple decision trees are created. Each one is trained with a different random subset of the data, and a different random subset of the features. You can see above that we specify 100 trees, which is the default.

Then the Random Forest model simply takes the average of all its trees to product the final result.

In [35]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [36]:
Tester.evaluate(random_forest, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$79 $74 $4 $8 $121 $150 $75 $69 $34 $242 $517 $294 $55 $61 $3 $2 $55 $85 $39 $52 $16 $25 $18 $30 $143 $321 $157 $117 $25 $51 $43 $94 $60 $16 $29 $487 $47 $36 $141 $65 $145 $59 $13 $53 $132 $41 $60 $55 $56 $6 $15 $56 $235 $34 $295 $47 $51 $154 $4 $44 $110 $9 $58 $13 $465 $11 $23 $202 $14 $131 $10 $20 $35 $137 $1 $78 $112 $23 $28 $85 $69 $72 $41 $49 $12 $28 $13 $110 $8 $113 $29 $166 $2 $84 $52 $48 $11 $55 $133 $293 $34 $2 $7 $65 $37 $32 $121 $202 $15 $179 $37 $57 $69 $30 $57 $64 $92 $14 $216 $29 $22 $29 $46 $45 $35 $17 $46 $66 $88 $18 $23 $42 $70 $22 $47 $56 $28 $5 $67 $39 $0 $151 $7 $111 $108 $64 $29 $314 $85 $5 $22 $104 $56 $76 $42 $78 $106 $27 $50 $11 $10 $7 $3 $29 $382 $43 $267 $30 $18 $36 $21 $11 $236 $33 $15 $59 $56 $2 $39 $6 $339 $15 $38 $83 $123 $48 $58 $73 $13 $28 $38 $53 $25 $6 $7 $19 $68 $42 $1 $16 

In [37]:
# This is how to save the model if you want to, particularly if you run this on a larger dataset

# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

## Introducing XGBoost

Like Random Forest, XGBoost is also an ensemble model that combines multiple decision trees.

But unlike Random Forest, XGBoost builds one tree after another, with each next tree correcting for errors in the prior trees, using 'gradient descent'.

It's much faster than Random Forest, so we can run it for the full dataset, and it's typically better at generalizing.

**If this import doesn't work, please skip this! It's not required. On a Mac, you might need to do `brew install libomp` in the terminal.**

In [40]:
import xgboost as xgb

In [ ]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [45]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [46]:
Tester.evaluate(xg_boost, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$95 $91 $24 $12 $167 $111 $18 $20 $22 $105 $359 $201 $138 $131 $16 $16 $44 $13 $31 $48 $4 $74 $14 $46 $171 $262 $146 $35 $50 $39 $78 $86 $23 $22 $33 $303 $19 $48 $163 $82 $130 $63 $54 $56 $126 $31 $73 $48 $35 $13 $23 $74 $133 $5 $61 $91 $54 $137 $17 $12 $28 $10 $46 $1 $408 $91 $6 $251 $58 $140 $18 $41 $79 $145 $8 $74 $173 $19 $34 $69 $74 $54 $39 $25 $21 $65 $52 $132 $13 $183 $23 $91 $36 $20 $28 $75 $26 $39 $153 $158 $32 $42 $3 $55 $3 $48 $51 $248 $31 $70 $39 $18 $62 $67 $43 $23 $43 $25 $53 $58 $59 $147 $86 $20 $86 $25 $5 $114 $136 $15 $27 $50 $90 $24 $72 $89 $84 $38 $43 $59 $15 $130 $40 $64 $49 $67 $37 $282 $77 $27 $19 $162 $22 $53 $30 $119 $128 $16 $14 $5 $99 $26 $5 $9 $391 $2 $112 $50 $16 $10 $29 $16 $197 $27 $30 $49 $27 $0 $31 $24 $253 $14 $189 $19 $157 $76 $56 $73 $21 $34 $89 $68 $1 $56 $10 $59 $12 $9 $36 $23 

In [51]:
import plotly.graph_objects as go

results = [
    ("Constant", "green", 106.18),
    ("Linear Regression", "green", 101.56),
    ("NLP + LR", "green", 76.81),
    ("Random Forest", "green", 72.28),
    ("XGBoost", "green", 68.23),
]

labels, colors, values = zip(*results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="Prediction error from each model",
    yaxis = dict(range=[0, max(values)], title="Error"),
    xaxis=dict(tickangle=-45),
    width = 1000,
    height=800
)

fig.show()